# Ablation Visualization
Side-by-side comparison of all ablation checkpoints on a single validation frame.
Two visualization modes:
- **BEV** — top-down radar point cloud with 3D bounding boxes
- **Camera 2D** — projected 3D boxes drawn on the camera image

In [ ]:
import os, sys

# Run from project root or adjust this path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
from omegaconf import OmegaConf, DictConfig

from vod.configuration import KittiLocations
from vod.frame import FrameDataLoader, FrameTransformMatrix

from src.model.detector import CenterPoint
from src.dataset import ViewOfDelft, collate_vod_batch

## Configuration

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
FRAME_IDX   = 0       # Index into the validation split (0-based)
DATA_ROOT   = 'data/view_of_delft'
SCORE_THRESH = 0.2

# ── Ablation registry ─────────────────────────────────────────────────────────
ABLATIONS = [
    ('A0 Baseline',         'outputs/A0/checkpoints/last.ckpt'),
    ('A1 PointPainting',    'outputs/A1/checkpoints/last.ckpt'),
    ('A2 +Doppler Cluster', 'outputs/A2/checkpoints/last.ckpt'),
    ('A3 +Doppler Attn',    'outputs/A3/checkpoints/last.ckpt'),
    ('A4 +Both Doppler',    'outputs/A4/checkpoints/last.ckpt'),
    ('A5 +5 Frames',        'outputs/A5/checkpoints/last.ckpt'),
    ('A6 +Neck Refine',     'outputs/A6/checkpoints/last.ckpt'),
    ('A7 +Wide PFN',        'outputs/A7/checkpoints/last.ckpt'),
]

CLASS_NAMES  = ['Car', 'Pedestrian', 'Cyclist']
CLASS_COLORS = ['#FF4444', '#44FF44', '#4488FF']   # red, green, blue

# BEV view limits (metres, radar frame)
X_MIN, X_MAX =  0.0,  51.2
Y_MIN, Y_MAX = -25.6,  25.6

## Geometry helpers

In [ ]:
def box_corners_bev(box):
    """Return (4, 2) BEV corners for a box [x, y, z, l, w, h, yaw]."""
    x, y, l, w, yaw = box[0], box[1], box[3], box[4], box[6]
    half_l, half_w = l / 2.0, w / 2.0
    local = np.array([[ half_l,  half_w],
                      [ half_l, -half_w],
                      [-half_l, -half_w],
                      [-half_l,  half_w]])
    c, s = np.cos(yaw), np.sin(yaw)
    rot = np.array([[c, -s], [s, c]])
    world = local @ rot.T
    world[:, 0] += x
    world[:, 1] += y
    return world


def box_corners_3d(box):
    """Return (8, 3) corners in LiDAR frame for a box [x, y, z, l, w, h, yaw].
    Origin convention: (0.5, 0.5, 0) — z is the bottom of the box.
    """
    x, y, z, l, w, h, yaw = box
    half_l, half_w = l / 2.0, w / 2.0
    # bottom face then top face (z=0 → z=h relative to box bottom)
    local = np.array([
        [ half_l,  half_w, 0],
        [ half_l, -half_w, 0],
        [-half_l, -half_w, 0],
        [-half_l,  half_w, 0],
        [ half_l,  half_w, h],
        [ half_l, -half_w, h],
        [-half_l, -half_w, h],
        [-half_l,  half_w, h],
    ])
    c, s = np.cos(yaw), np.sin(yaw)
    rot = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])
    corners = local @ rot.T
    corners[:, 0] += x
    corners[:, 1] += y
    corners[:, 2] += z
    return corners


# Edges connecting the 8 corners of a 3D box
BOX_EDGES = [
    (0,1),(1,2),(2,3),(3,0),   # bottom face
    (4,5),(5,6),(6,7),(7,4),   # top face
    (0,4),(1,5),(2,6),(3,7),   # verticals
]

## Model / dataset helpers

In [ ]:
def build_dataset_from_ckpt(ckpt_cfg, data_root, split='val'):
    """Reconstruct the ViewOfDelft dataset from a checkpoint config."""
    dataset_cfg = {}
    if 'dataset' in ckpt_cfg:
        dataset_cfg = OmegaConf.to_container(ckpt_cfg['dataset'], resolve=True)
    if 'radar_mode' in ckpt_cfg and 'radar_mode' not in dataset_cfg:
        dataset_cfg['radar_mode'] = ckpt_cfg['radar_mode']
    return ViewOfDelft(data_root=data_root, split=split, **dataset_cfg)


def run_inference(model, batch, score_thresh):
    """Run the model on one batch; return list of dicts with boxes/scores/labels."""
    model.eval()
    with torch.no_grad():
        ret_dict, _ = model._model_forward_tta(batch['pts'])
        bbox_list = model.head.get_bboxes(ret_dict, img_metas=batch['metas'])
    results = []
    for bboxes, scores, labels in bbox_list:
        keep = scores >= score_thresh
        results.append(dict(
            boxes  = bboxes[keep].tensor.cpu().numpy(),
            scores = scores[keep].cpu().numpy(),
            labels = labels[keep].cpu().numpy().astype(int),
        ))
    return results


def load_ablation(name, ckpt_path, data_root, frame_idx, score_thresh):
    """Load one checkpoint, run inference on frame_idx, return (pts_np, gt_boxes, gt_labels, pred, num_frame, transforms)."""
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    ckpt_cfg = DictConfig(ckpt['hyper_parameters']['config'])

    dataset = build_dataset_from_ckpt(ckpt_cfg, data_root, split='val')
    if frame_idx >= len(dataset):
        print(f'  frame_idx {frame_idx} out of range (dataset has {len(dataset)} samples)')
        return None

    sample = dataset[frame_idx]
    batch  = collate_vod_batch([sample])
    num_frame = batch['metas'][0]['num_frame']

    pts_np    = batch['pts'][0].numpy()
    gt_boxes  = batch['gt_bboxes_3d'][0].tensor.numpy()
    gt_labels = batch['gt_labels_3d'][0].numpy()

    model = CenterPoint.load_from_checkpoint(ckpt_path, map_location='cuda')
    model.cuda().eval()

    batch_gpu = dict(
        pts          = [torch.tensor(pts_np).cuda()],
        gt_bboxes_3d = batch['gt_bboxes_3d'],
        gt_labels_3d = batch['gt_labels_3d'],
        metas        = batch['metas'],
    )
    try:
        pred = run_inference(model, batch_gpu, score_thresh)[0]
    except Exception as e:
        print(f'  Inference failed: {e}')
        pred = None

    del model
    torch.cuda.empty_cache()

    # Load camera transforms for 2D projection
    kitti_locs = KittiLocations(root_dir=data_root)
    frame_data = FrameDataLoader(kitti_locations=kitti_locs, frame_number=num_frame)
    transforms = FrameTransformMatrix(frame_data)

    return dict(
        pts_np=pts_np, gt_boxes=gt_boxes, gt_labels=gt_labels,
        pred=pred, num_frame=num_frame,
        frame_data=frame_data, transforms=transforms,
    )

## Visualization: BEV (Bird's Eye View)

In [ ]:
def draw_bev_box(ax, corners, color, linewidth=1.5, linestyle='-', alpha=1.0, zorder=3):
    poly = plt.Polygon(np.vstack([corners, corners[0]]),
                       closed=True, fill=False,
                       edgecolor=color, linewidth=linewidth,
                       linestyle=linestyle, alpha=alpha, zorder=zorder)
    ax.add_patch(poly)
    front_mid = (corners[0] + corners[1]) / 2.0
    center    = corners.mean(axis=0)
    ax.annotate('', xy=front_mid, xytext=center,
                arrowprops=dict(arrowstyle='->', color=color,
                                lw=linewidth, mutation_scale=8),
                zorder=zorder + 1)


def plot_bev(ax, pts, pred, gt_boxes, gt_labels, title):
    """Fill one BEV subplot with radar points and bounding boxes."""
    ax.set_facecolor('#1a1a2e')
    ax.set_xlim(X_MIN, X_MAX)
    ax.set_ylim(Y_MIN, Y_MAX)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=8, color='white', pad=3)
    ax.tick_params(colors='#888888', labelsize=6)
    for spine in ax.spines.values():
        spine.set_edgecolor('#444444')

    # Radar points — colour by Doppler if channel 5 exists
    if pts.shape[1] > 5:
        doppler = pts[:, 5]
        vmax = max(abs(doppler).max(), 0.1)
        ax.scatter(pts[:, 0], pts[:, 1],
                   c=doppler, cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                   s=2, alpha=0.6, linewidths=0, zorder=2)
    else:
        ax.scatter(pts[:, 0], pts[:, 1],
                   c='#aaaaaa', s=2, alpha=0.6, linewidths=0, zorder=2)

    # Ground truth — dashed white
    for box, label in zip(gt_boxes, gt_labels):
        corners = box_corners_bev(box)
        draw_bev_box(ax, corners, color='white', linewidth=1.2,
                     linestyle='--', alpha=0.85, zorder=4)

    # Predictions — solid class colour
    if pred is not None:
        for box, label, score in zip(pred['boxes'], pred['labels'], pred['scores']):
            if label >= len(CLASS_COLORS):
                continue
            corners = box_corners_bev(box)
            draw_bev_box(ax, corners, color=CLASS_COLORS[label], linewidth=1.5,
                         linestyle='-', alpha=0.9, zorder=5)
            cx, cy = box[0], box[1]
            if X_MIN < cx < X_MAX and Y_MIN < cy < Y_MAX:
                ax.text(cx, cy, f'{score:.2f}', fontsize=4,
                        color=CLASS_COLORS[label],
                        ha='center', va='center', zorder=6)

    ax.plot(0, 0, 'w^', markersize=4, zorder=7)


def visualize_bev(frame_idx=FRAME_IDX, score_thresh=SCORE_THRESH,
                  data_root=DATA_ROOT, out=None):
    """BEV comparison of all ablations on a single validation frame."""
    available = [(n, c) for n, c in ABLATIONS if os.path.isfile(c)]
    if not available:
        print('No checkpoints found — check ABLATIONS paths.')
        return
    print(f'Found {len(available)} checkpoints: {[n for n, _ in available]}')

    n     = len(available)
    ncols = min(n, 4)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(ncols * 4.5, nrows * 4.5),
                             facecolor='#0d0d1a')
    fig.subplots_adjust(wspace=0.05, hspace=0.15)
    axes_flat = np.array(axes).flatten() if n > 1 else [axes]
    for ax in axes_flat[n:]:
        ax.set_visible(False)

    for plot_idx, (name, ckpt_path) in enumerate(available):
        print(f'\n[{plot_idx+1}/{n}] {name} ...')
        result = load_ablation(name, ckpt_path, data_root, frame_idx, score_thresh)
        if result is None:
            continue
        plot_bev(axes_flat[plot_idx],
                 result['pts_np'], result['pred'],
                 result['gt_boxes'], result['gt_labels'],
                 title=f"{name}  |  frame {result['num_frame']}")

    legend_handles = [mpatches.Patch(facecolor='none', edgecolor='white',
                                     linestyle='--', label='Ground truth')]
    for nm, col in zip(CLASS_NAMES, CLASS_COLORS):
        legend_handles.append(mpatches.Patch(facecolor='none', edgecolor=col,
                                             label=f'Pred: {nm}'))
    fig.legend(handles=legend_handles, loc='lower center',
               ncol=len(CLASS_NAMES) + 1, fontsize=8, framealpha=0.3,
               facecolor='#0d0d1a', labelcolor='white',
               bbox_to_anchor=(0.5, 0.0))
    fig.suptitle(f'BEV Ablation Comparison — val frame idx {frame_idx}',
                 color='white', fontsize=12, y=1.01)

    output_path = out or f'ablation_bev_frame{frame_idx:04d}.png'
    plt.savefig(output_path, dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.show()
    print(f'\nSaved to {output_path}')

## Visualization: 2D Camera Frame

In [ ]:
def project_boxes_to_image(boxes, transforms):
    """Project 3D boxes (N, 7) from LiDAR frame to image pixel coords.
    Returns list of (8, 2) pixel arrays, one per box. Boxes with any
    corner behind the camera are returned as None.
    """
    t_cam_lidar = transforms.t_camera_lidar      # (4, 4)
    proj_mat    = transforms.camera_projection_matrix  # (3, 4)
    pixel_corners = []
    for box in boxes:
        corners = box_corners_3d(box)             # (8, 3)
        homo    = np.hstack([corners, np.ones((8, 1))])  # (8, 4)
        cam     = (t_cam_lidar @ homo.T).T        # (8, 4) in camera frame
        if (cam[:, 2] <= 0).any():                # any corner behind camera
            pixel_corners.append(None)
            continue
        img_homo = (proj_mat @ cam.T).T           # (8, 3)
        u = img_homo[:, 0] / img_homo[:, 2]
        v = img_homo[:, 1] / img_homo[:, 2]
        pixel_corners.append(np.stack([u, v], axis=1))  # (8, 2)
    return pixel_corners


def draw_3d_box_on_image(ax, pixel_corners, color, linewidth=1.5,
                         linestyle='-', alpha=1.0):
    """Draw the 12 edges of a projected 3D box on an image axes."""
    if pixel_corners is None:
        return
    for i, j in BOX_EDGES:
        ax.plot([pixel_corners[i, 0], pixel_corners[j, 0]],
                [pixel_corners[i, 1], pixel_corners[j, 1]],
                color=color, linewidth=linewidth,
                linestyle=linestyle, alpha=alpha)


def plot_camera_2d(ax, image, pred, gt_boxes, gt_labels, transforms, title):
    """Fill one camera subplot with the RGB image and projected 3D boxes."""
    ax.imshow(image)
    ax.set_title(title, fontsize=8, pad=3)
    ax.axis('off')

    # Ground truth — dashed white
    gt_pixels = project_boxes_to_image(gt_boxes, transforms)
    for corners in gt_pixels:
        draw_3d_box_on_image(ax, corners, color='white',
                             linewidth=1.2, linestyle='--', alpha=0.85)

    # Predictions — solid class colour
    if pred is not None:
        pred_pixels = project_boxes_to_image(pred['boxes'], transforms)
        for corners, label, score in zip(pred_pixels, pred['labels'], pred['scores']):
            if label >= len(CLASS_COLORS) or corners is None:
                continue
            draw_3d_box_on_image(ax, corners, color=CLASS_COLORS[label],
                                 linewidth=1.5, linestyle='-', alpha=0.9)
            cx = corners[:, 0].mean()
            cy = corners[:, 1].mean()
            ax.text(cx, cy, f'{score:.2f}', fontsize=4,
                    color=CLASS_COLORS[label], ha='center', va='center')


def visualize_camera_2d(frame_idx=FRAME_IDX, score_thresh=SCORE_THRESH,
                        data_root=DATA_ROOT, out=None):
    """Camera-frame 2D comparison of all ablations on a single validation frame."""
    available = [(n, c) for n, c in ABLATIONS if os.path.isfile(c)]
    if not available:
        print('No checkpoints found — check ABLATIONS paths.')
        return
    print(f'Found {len(available)} checkpoints: {[n for n, _ in available]}')

    n     = len(available)
    ncols = min(n, 4)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(ncols * 5.5, nrows * 3.5))
    fig.subplots_adjust(wspace=0.02, hspace=0.18)
    axes_flat = np.array(axes).flatten() if n > 1 else [axes]
    for ax in axes_flat[n:]:
        ax.set_visible(False)

    for plot_idx, (name, ckpt_path) in enumerate(available):
        print(f'\n[{plot_idx+1}/{n}] {name} ...')
        result = load_ablation(name, ckpt_path, data_root, frame_idx, score_thresh)
        if result is None:
            continue
        image = result['frame_data'].image
        plot_camera_2d(axes_flat[plot_idx], image,
                       result['pred'], result['gt_boxes'], result['gt_labels'],
                       result['transforms'],
                       title=f"{name}  |  frame {result['num_frame']}")

    legend_handles = [mpatches.Patch(facecolor='none', edgecolor='white',
                                     linestyle='--', label='Ground truth')]
    for nm, col in zip(CLASS_NAMES, CLASS_COLORS):
        legend_handles.append(mpatches.Patch(facecolor='none', edgecolor=col,
                                             label=f'Pred: {nm}'))
    fig.legend(handles=legend_handles, loc='lower center',
               ncol=len(CLASS_NAMES) + 1, fontsize=8,
               bbox_to_anchor=(0.5, 0.0))
    fig.suptitle(f'Camera 2D Ablation Comparison — val frame idx {frame_idx}',
                 fontsize=12, y=1.01)

    output_path = out or f'ablation_cam2d_frame{frame_idx:04d}.png'
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\nSaved to {output_path}')

## Run — BEV visualization

In [ ]:
visualize_bev(frame_idx=FRAME_IDX, score_thresh=SCORE_THRESH, data_root=DATA_ROOT)

## Run — Camera 2D visualization

In [ ]:
visualize_camera_2d(frame_idx=FRAME_IDX, score_thresh=SCORE_THRESH, data_root=DATA_ROOT)